**PHASE 3. LLM INTEGRATION**

As stated in the previous phase, phase 3 will integrate LLM-explanatory systems which aim to establish comprehension for non-technical users. This notebook serves as the analytical layer in the LLM system, which will pass analytical informations (e.g., rolling volatility, monthly average, and trend) to the LLM, using finalized model from the previous phase. All required informations are listed in the schema, which can be accessed via BTC-Volatility-Forecast/analysis/analysis_schema.md in the GitHub repository.

**I. Preparation**

Below are the used libraries required for analysis.

In [ ]:
!pip install arch #Since arch module is not inherently installed, we must install it manually.
from arch import arch_model
import yfinance as yf
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
from scipy import stats
from datetime import datetime

**II. ForecastInput**

This class (or section) is designed to establish transparency regarding how the model calculates the next-day price volatility. This includes the source of bitcoin price data, and the feature info (i.e. feature values and importances).

In [ ]:
class ForecastInputs:
    def __init__(self):
        self.btc = yf.download("BTC-USD", start="2018-01-01")
        self.btc = self.btc[["Close"]].dropna()
        log_diff = np.log(self.btc["Close"]).diff().dropna()
        res = arch_model(log_diff * 10, vol="GARCH", mean="ARX", lags=1, p=1, q=1, dist="t").fit(disp="off")

        self.btc = self.btc.iloc[1:].copy()
        self.btc["log_return"] = log_diff.values
        self.btc["garch_volatility"] = res.conditional_volatility / 10

        self.btc["return_lag_1"] = self.btc["log_return"].shift(1)
        self.btc["return_lag_2"] = self.btc["log_return"].shift(2)

        self.btc["garch_vol_lag_1"] = self.btc["garch_volatility"].shift(1)
        self.btc["garch_vol_lag_2"] = self.btc["garch_volatility"].shift(2)

        self.btc["rolling_mean_7"] = self.btc["log_return"].rolling(7).mean()

        self.btc["rolling_std_3"] = self.btc["log_return"].rolling(3).std()

        self.btc["rolling_std_7"] = self.btc["log_return"].rolling(7).std()

        self.btc["rolling_std_14"] = self.btc["log_return"].rolling(14).std()

        self.btc["target_volatility"] = self.btc["garch_volatility"].shift(-1)

        self.features = [
            "return_lag_1",
            "return_lag_2",
            "garch_vol_lag_1",
            "garch_vol_lag_2",
            "rolling_mean_7",
            "rolling_std_3",
            "rolling_std_7",
            "rolling_std_14"
        ]
        self.all_time_volatility = self.btc["garch_volatility"].dropna()
        self.all_time_features_data = self.btc[self.features].dropna().copy()
        self.last_feature_data = self.all_time_features_data.iloc[-1:]
        self.btc = self.btc.dropna().copy()

    def retrieve_features(self):
        self.feature_values = dict()
        for feature in self.features:
            self.feature_values[feature] = float(self.last_feature_data[feature].values[0])
        return self.feature_values

    def retrieve_feature_importance(self):
        if pred.model is None:
            pred.train()
        if pred.prediction is None:
            pred.predict()
        self.model = pred.model
        self.feature_importance = dict()
        for i, feature in enumerate(self.features):
            self.feature_importance[feature] = float(self.model.feature_importances_[i]) * 100
        return self.feature_importance

**III. Prediction**

This class comprises the entire prediction processes, including training and forecasting. There are two supplementary informations to be provided for the users, including:
1. Risk level: This is designed to provide brief sense on how investors (BTC traders) should respond to the volatility. Risk is measured in a relative sense, where the risk level is considered 'high' if the predicted volatility is at the top 33% volatility, 'low' if the prediction is at the bottom 33%, and 'medium' if otherwise.

2. Volatility regime: This provides a sense whether the recent (last 30 days) volatility remains high or low. This is also measured in a relative sense, where if the volatility regime is considered 'high' if the average monthly volatility is above the top 33% cutoff (from the historical monthly volatility averages starting from 2018), 'low' if below the bottom 33% cutoff, and 'medium' if otherwise.

In [ ]:
class Prediction:
    def __init__(self):
        self.X = forecast_input.btc[forecast_input.features]
        self.y = forecast_input.btc["target_volatility"]
        self.monthly_avg = (
            forecast_input.all_time_volatility
            .iloc[-30:]
            .mean()
        )
        self.model = None
        self.pipeline = None
        self.prediction = None
        self.daily_threshold = None
        self.monthly_threshold = None

    def train(self):
        self.model = RandomForestRegressor(
            n_estimators = 100,
            max_depth = 5,
            random_state = 123
        )

        self.pipeline = Pipeline([
            ("scaler", StandardScaler())
        ])

        self.X_scaled = self.pipeline.fit_transform(self.X)
        self.model.fit(self.X_scaled, self.y)

    def predict(self):
        if self.model is None or self.pipeline is None:
            self.train()
        last_scaled = self.pipeline.transform(forecast_input.last_feature_data)
        self.prediction = dict()
        self.prediction = float(self.model.predict(last_scaled)[0])
        if self.prediction >= float(forecast_input.all_time_volatility.iloc[-1]):
            self.direction = "higher"
        else:
            self.direction = "lower"
        return self.prediction

    def calculate_thresholds(self, data): #Calculate 33rd and 67th percentile
        thresholds = [np.percentile(data, 33), np.percentile(data, 67)]
        return thresholds

    def risk_level_judgement(self):
        if self.prediction is None:
            self.predict()
        if self.daily_threshold is None:
            self.daily_threshold = self.calculate_thresholds(forecast_input.all_time_volatility)
        if self.prediction > self.daily_threshold[1]:
            self.risk_level = "high"
        elif self.prediction < self.daily_threshold[0]:
            self.risk_level = "low"
        else:
            self.risk_level = "medium"
        return self.risk_level

    def vol_regime_judgement(self):
        if self.prediction is None:
            self.predict()
        if self.monthly_threshold is None:
            self.monthly_threshold = self.calculate_thresholds(forecast_input.all_time_volatility.rolling(30).mean().dropna())
        if self.monthly_avg < self.monthly_threshold[0]:
            self.vol_regime_level = "low"
        elif self.monthly_avg > self.monthly_threshold[1]:
            self.vol_regime_level = "high"
        else:
            self.vol_regime_level = "medium"

        self.vol_regime_percentile = stats.percentileofscore(forecast_input.all_time_volatility.rolling(30).mean().dropna(), self.monthly_avg)
        return {'vol_regime_level': self.vol_regime_level, 'vol_regime_percentile': float(self.vol_regime_percentile)}

**IV. HistAnalysis**

This corresponds to the historical_analysis section of the schema, which details the behaviour of bitcoin price volatility in the last 30 days (excluding predicted volatility). This includes monthly average, monthly maximum volatility (monthly_max), monthly minimum volatility (monthly_min), and the analysis of monthly trend and volatility persistence.

The trend comprises the trend direction (up/stable/down) and trend strength, where trend strength is measured in a relative sense. The relative benchmarks follow the risk level and volatility regime benchmarks in the prediction section.

Persistence is measured by calculating the volatility autocorrelation with lag = 1 (in other words, the correlation between last 30 days volatilities with the same data but shifted 1 day to the past). The persistence level is also categorized in a relative sense, whose benchmarks follow the usual convention.



In [ ]:
class HistAnalysis:
  def __init__(self):
    self.monthly_volatility = forecast_input.all_time_volatility.iloc[-30:]
    self.monthly_avg = self.monthly_volatility.mean()
    self.monthly_max = max(self.monthly_volatility)
    self.monthly_min = min(self.monthly_volatility)

  def trend_analysis(self):
    days = np.array([i for i in range(len(self.monthly_volatility))])
    i = 0
    self.slopes = list()
    while i + 29 < len(forecast_input.all_time_volatility):
      self.slopes.append(abs(np.polyfit(days, np.array(forecast_input.all_time_volatility.iloc[i:i+30]), deg = 1)[0]))
      i += 1
    self.trend = dict()
    self.slope = np.polyfit(days, np.array(self.monthly_volatility), deg = 1)[0]
    if self.slope > 0:
      self.trend['trend'] = "increasing"
    else:
      self.trend['trend'] = "decreasing"

    self.monthly_percentile = stats.percentileofscore(self.slopes[:-1], self.slopes[-1])
    if self.monthly_percentile > 67:
      self.trend['trend_strength'] = "strong"
    elif self.monthly_percentile < 33:
      if self.monthly_percentile < 17:
        self.trend['trend'] = "stable"
      else:
        self.trend['trend_strength'] = "weak"
    else:
      self.trend['trend_strength'] = "medium"
    return self.trend

  def persistence_analysis(self):
    self.persistence_index = float(self.monthly_volatility.autocorr(lag = 1))
    self.persistence = {'persistence_index': self.persistence_index}
    if self.persistence_index > 0.67:
      self.persistence['persistence_level'] = "strong"
    elif self.persistence_index < 0.33:
      self.persistence['persistence_level'] = "weak"
    else:
      self.persistence['persistence_level'] = "medium"
    return self.persistence

**V. Confidence**

This section measures the certainty index of the prediction, which is measured based on persistence and tree disagreement percentile (ensemble uncertainty). The confidence index is ranged from -30 to 100. Confidence index is 'high' if more than 67, 'low' if less than 33, 'medium' if otherwise.

Tree disagreement measures the standardized spread of individual tree predictions (since we are using Random Forest Regression). Tree disagreement is calculated by dividing the standard deviation of individual predictions of every tree by the average of the individual predictions (which is the predicted data). The higher the tree disagreement, the lower the confidence index.

In [ ]:
class Confidence:
  def __init__(self):
    self.confidence_index = None
    self.confidence_level = None

  def confidence_analysis(self):
    if pred.prediction is None:
      pred.predict()

    self.persistence = hist_analysis.persistence_analysis()['persistence_index'] * 100

    self.tree_disagreements = list()
    for i in range(len(forecast_input.all_time_features_data)):
      sample = forecast_input.all_time_features_data.iloc[i:i+1]
      sample = pred.pipeline.transform(sample)
      self.prediction = float(pred.model.predict(sample)[0])
      self.individual_preds = np.array([float(tree.predict(sample)[0]) for tree in pred.model.estimators_])
      self.tree_disagreements.append(np.std(self.individual_preds) / abs(self.prediction))
    self.tree_disagreements_percentile = stats.percentileofscore(self.tree_disagreements[:-1], self.tree_disagreements[-1])

    self.confidence_index = 0.7 * (100 - self.tree_disagreements_percentile) + 0.3 * self.persistence

    if self.confidence_index > 67:
      self.confidence_level = "high"
    elif self.confidence_index < 33:
      self.confidence_level = "low"
    else:
      self.confidence_level = "medium"

    return {'confidence_level': self.confidence_level, 'confidence_index': self.confidence_index}

**VI. Implementation**

We have defined all classes required for analysis. The analysis_output variable is consistent with the defined schema and will be passed into the LLM to display statistical and analytical informations, and establish user-friendly natural-language-based explanation.

In [ ]:
forecast_input = ForecastInputs()
pred = Prediction()
hist_analysis = HistAnalysis()
confidence = Confidence()

analysis_output = {"prediction": {"predicted_volatility": pred.predict(),
                                  "forecast_relativity": pred.direction,
                                  "risk_level": pred.risk_level_judgement(),
                                  "volatility_regime": pred.vol_regime_judgement()},
                   "forecast_inputs": {"today_garch_volatility": float(forecast_input.all_time_volatility.iloc[-1]),
                                       "rolling_volatility": float(forecast_input.all_time_features_data['rolling_std_14'].iloc[-1]),
                                       "yesterday_volatility": float(forecast_input.all_time_volatility.iloc[-2]),
                                       "feature_values": forecast_input.retrieve_features(),
                                       "feature_importance": forecast_input.retrieve_feature_importance()},
                   "historical_analysis": {"monthly_average": float(hist_analysis.monthly_avg),
                                           "monthly_max": hist_analysis.monthly_max,
                                           "monthly_min:": hist_analysis.monthly_min,
                                           "trend": hist_analysis.trend_analysis(),
                                           "persistence": hist_analysis.persistence_analysis()},
                   "confidence": {"confidence_level": confidence.confidence_analysis()['confidence_level'],
                                  "confidence_index": float(confidence.confidence_index),
                                  "volatility_regime_percentile": float(pred.vol_regime_percentile),
                                  "persistence_percentile": confidence.persistence,
                                  "ensemble_uncertainty_percentile": float(confidence.tree_disagreements_percentile)},
                   "metadata": {"timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                                "model_version": "1.0"}
                   }
analysis_output
